# 03 · Gold — Business-Ready Aggregates

Join and aggregate silver into **analytics-ready** tables that answer business
questions directly. (Brief **§3→§4**.)

| table | grain | used for |
|---|---|---|
| `daily_consumption` | household × day | master fact (ML features) |
| `daily_totals` | day | demand trend, weather correlation |
| `acorn_profile` | acorn group × tariff | who consumes most |
| `load_profile` | half-hour slot | the 48-point daily demand curve |

In [ ]:
%run ./00_config_and_setup

In [ ]:
from pyspark.sql import functions as F

def save_gold(df, name):
    tgt = table("gold", name)
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true").saveAsTable(tgt)
    print(f"gold {name:<18} rows={spark.table(tgt).count():>12,}")
    return tgt

## 1. `daily_consumption` — master fact
derived `daily` ⨝ `households` (tariff/ACORN) ⨝ `weather` (by date) ⨝ `holidays`,
enriched with calendar features. Powers both the analysis and the ML.

In [ ]:
daily      = spark.table(table("silver", "daily"))
households = spark.table(table("silver", "households")).select("LCLid", "tariff", "acorn_group", "acorn_code")
weather    = spark.table(table("silver", "weather_daily"))
holidays   = spark.table(table("silver", "bank_holidays")).select("date", "holiday_name")

dc = (daily.join(households, "LCLid", "left")
            .join(weather, daily["day"] == weather["date"], "left").drop("date")
            .join(holidays, daily["day"] == holidays["date"], "left").drop("date"))

dc = (dc
    .withColumn("year",        F.year("day"))
    .withColumn("month",       F.month("day"))
    .withColumn("day_of_week", F.dayofweek("day"))          # 1=Sun ... 7=Sat
    .withColumn("is_weekend",  F.col("day_of_week").isin(1, 7).cast("int"))
    .withColumn("is_holiday",  F.col("holiday_name").isNotNull().cast("int"))
    .withColumn("season", F.when(F.col("month").isin(12, 1, 2), "Winter")
                           .when(F.col("month").isin(3, 4, 5),  "Spring")
                           .when(F.col("month").isin(6, 7, 8),  "Summer")
                           .otherwise("Autumn"))
    .withColumnRenamed("energy_sum", "daily_kwh"))

save_gold(dc, "daily_consumption")

## 2. `daily_totals` — one row per day

In [ ]:
totals = (spark.table(table("gold", "daily_consumption"))
    .groupBy("day", "season", "is_weekend", "is_holiday")
    .agg(F.round(F.sum("daily_kwh"), 1).alias("total_kwh"),
         F.round(F.avg("daily_kwh"), 3).alias("avg_kwh_per_home"),
         F.countDistinct("LCLid").alias("active_homes"),
         F.round(F.first("temp_avg"), 2).alias("temp_avg"))
    .orderBy("day"))

save_gold(totals, "daily_totals")

## 3. `acorn_profile` — consumption by affluence group & tariff

In [ ]:
profile = (spark.table(table("gold", "daily_consumption"))
    .groupBy("acorn_group", "tariff")
    .agg(F.round(F.avg("daily_kwh"), 3).alias("avg_daily_kwh"),
         F.round(F.expr("percentile_approx(daily_kwh, 0.5)"), 3).alias("median_daily_kwh"),
         F.countDistinct("LCLid").alias("households"))
    .orderBy(F.col("avg_daily_kwh").desc()))

save_gold(profile, "acorn_profile")

## 4. `load_profile` — average 48-point daily demand curve
Built straight from the silver half-hourly readings: average energy per half-hour-of-day.

In [ ]:
load_profile = (spark.table(table("silver", "halfhourly"))
    .groupBy("half_hour")
    .agg(F.round(F.avg("energy_kwh"), 5).alias("avg_kwh"))
    .withColumn("clock_time",
                F.format_string("%02d:%s", (F.col("half_hour") / 2).cast("int"),
                                F.when(F.col("half_hour") % 2 == 1, "30").otherwise("00")))
    .orderBy("half_hour"))

save_gold(load_profile, "load_profile")

In [ ]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD}"))
display(spark.table(table("gold", "daily_consumption")).limit(5))